# Find whistle intervals and slant delays

This notebook only analyzes an existing WAV: load → detect intervals → build contour templates → matched filter → cancel direct path → detect echoes. To create the default delayed test WAV, first run `Synthetic_Slant_Delay_Validation.ipynb`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, display
from scipy.io import wavfile

from Slant_delay_utills import AnalysisConfig, analyze_detections, detect_whistles


def plot_contours_and_templates(recording_analysis, title_prefix):
    if not recording_analysis.whistles:
        display("No accepted contours or templates.")
        return
    for whistle_index, result in enumerate(recording_analysis.whistles, start=1):
        detection = result.detection
        template_time = detection.start_seconds + np.arange(result.template.size) / recording_analysis.sample_rate
        template_phase = np.unwrap(np.angle(result.template))
        template_frequency = np.gradient(template_phase) * recording_analysis.sample_rate / (2 * np.pi)
        figure, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True, constrained_layout=True)
        axes[0].plot(detection.time_seconds, detection.frequency_hz / 1e3, color="0.75", label="Interpolated/support path")
        axes[0].scatter(detection.time_seconds[detection.observed_mask], detection.frequency_hz[detection.observed_mask] / 1e3, s=8, label="Observed ridge")
        axes[0].set(ylabel="Frequency (kHz)", title="Detected contour")
        axes[0].legend()
        axes[1].plot(template_time, template_frequency / 1e3)
        axes[1].set(xlabel="Recording time (s)", ylabel="Frequency (kHz)", title="Frequency encoded by the matched-filter template")
        figure.suptitle(f"{title_prefix} whistle {whistle_index}")
        plt.show()


def plot_matched_filter_responses(recording_analysis, config, title_prefix):
    if not recording_analysis.whistles:
        display("No matched-filter response because no whistle was accepted.")
        return
    for whistle_index, result in enumerate(recording_analysis.whistles, start=1):
        relative_lag_ms = (result.lags - result.lags[result.direct_index]) / recording_analysis.sample_rate * 1e3
        scale = max(np.abs(result.response).max(), np.finfo(float).eps)
        figure, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True, constrained_layout=True)
        for axis, response, title in ((axes[0], result.response, "Original matched-filter response"), (axes[1], result.residual_response, "After direct-path cancellation")):
            axis.plot(relative_lag_ms, np.abs(response) / scale)
            axis.axvline(0, color="black", linestyle="--", label="Direct path")
            for echo_number, (echo_index, delay) in enumerate(zip(result.echo_indices, result.delay_seconds), start=1):
                axis.axvline(relative_lag_ms[echo_index], color=f"C{echo_number}", linestyle="--", label=f"Echo {echo_number}: {delay * 1e3:.3f} ms")
            axis.set(ylabel="Normalized magnitude", title=title)
        axes[0].legend()
        margin_ms = max(1.0, config.min_delay_seconds * 1e3)
        axes[1].set(xlim=(-margin_ms, config.max_delay_seconds * 1e3 + margin_ms), xlabel="Lag from direct arrival (ms)")
        figure.suptitle(f"{title_prefix} whistle {whistle_index}")
        plt.show()
        if result.delay_seconds.size == 0:
            display({"status": "No credible echo", "warnings": list(result.warnings)})

## 1. Select the WAV and analysis limits

In [ ]:
# Run Synthetic_Slant_Delay_Validation.ipynb first to generate this delayed fixture.
INPUT_WAV = Path("synthetic_dolphin_chirp.wav")
CONFIG = AnalysisConfig(
    whistle_band_hz=(3_500.0, 10_500.0),
    min_delay_seconds=0.002,
    max_delay_seconds=0.030,
    min_path_separation_seconds=0.002,
    max_echoes=4,
    output_dir=Path("whistle_analysis/unknown_wav"),
)

## 2. Load and play the recording

In [ ]:
sample_rate, audio = wavfile.read(INPUT_WAV)
display(Audio(audio, rate=sample_rate, normalize=True))
print(f"Loaded {INPUT_WAV}: {audio.shape[0] / sample_rate:.3f} s at {sample_rate} Hz")

## 3. Detect whistle intervals

In [ ]:
detections = detect_whistles(INPUT_WAV, CONFIG)
detection_rows = [{"whistle": index, "start_s": round(item.start_seconds, 4), "end_s": round(item.end_seconds, 4), "detection_confidence": round(item.detection_confidence, 3), "contour_confidence": round(item.contour_confidence, 3)} for index, item in enumerate(detections, start=1)]
display(detection_rows if detection_rows else "No credible whistles detected.")
mono = audio.astype(float) if audio.ndim == 1 else audio.astype(float).mean(axis=1)
recording_time = np.arange(mono.size) / sample_rate
figure, axis = plt.subplots(figsize=(12, 3), constrained_layout=True)
axis.plot(recording_time, mono, color="black", linewidth=0.5)
for index, detection in enumerate(detections, start=1):
    axis.axvspan(detection.start_seconds, detection.end_seconds, alpha=0.2, label=f"Whistle {index}")
axis.set(xlabel="Time (s)", ylabel="Amplitude", title="Automatically detected intervals")
if detections:
    axis.legend()
plt.show()

## 4. Build templates and estimate delays

In [ ]:
analysis = analyze_detections(INPUT_WAV, CONFIG, detections)
analysis_rows = [{"whistle": index, "delay_samples": result.delay_samples.tolist(), "delay_ms": [round(value * 1e3, 4) for value in result.delay_seconds], "delay_confidence": np.round(result.delay_confidence, 3).tolist(), "warnings": list(result.warnings)} for index, result in enumerate(analysis.whistles, start=1)]
display(analysis_rows if analysis_rows else "No intervals were available for delay analysis.")
print(f"Saved structured result to {analysis.output_dir / 'analysis.npz'}")

In [ ]:
GROUND_TRUTH_FILE = Path("synthetic_dolphin_chirp_ground_truth.npz")
if INPUT_WAV.resolve() == Path("synthetic_dolphin_chirp.wav").resolve() and GROUND_TRUTH_FILE.is_file():
    with np.load(GROUND_TRUTH_FILE, allow_pickle=False) as ground_truth:
        true_delay_samples = ground_truth["delay_samples"]
    calculated_delay_samples = analysis.whistles[0].delay_samples if len(analysis.whistles) == 1 else np.array([], dtype=int)
    errors = np.abs(calculated_delay_samples - true_delay_samples) if calculated_delay_samples.size == true_delay_samples.size else np.array([], dtype=int)
    ground_truth_passed = len(analysis.whistles) == 1 and calculated_delay_samples.size == true_delay_samples.size and np.all(errors <= 2)
    display({"ground_truth_samples": true_delay_samples.tolist(), "calculated_samples": calculated_delay_samples.tolist(), "absolute_error_samples": errors.tolist(), "result": "PASS" if ground_truth_passed else "FAIL"})
else:
    display("Ground-truth comparison skipped for this WAV.")

In [ ]:
plot_contours_and_templates(analysis, "Input WAV")

## 5. Inspect direct and reflected matched-filter responses

In [ ]:
plot_matched_filter_responses(analysis, CONFIG, "Input WAV")